In [26]:
from bs4 import BeautifulSoup

import requests

import pandas as pd

import time

import random


 # European Banking Authority

In [27]:
def polite_request(url, max_retries=5):
    delay = 2
    for attempt in range(max_retries):
        response = requests.get(url, headers=headers)
        

        if "Request Rate Threshold Exceeded" in response.text:
            print(f"Rate limit hit. Retrying in {delay} seconds...")
            time.sleep(delay)
            delay *= 2  
            continue
        elif response.status_code != 200:
            print(f"Unexpected status {response.status_code}. Retrying...")
            time.sleep(delay)
            delay *= 2
            continue
        else:
            return response
    raise Exception("Failed to retrieve page after several attempts.")

In [37]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,"
              "image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Referer": "https://www.google.com/",
    "DNT": "1", 
}

base_url = 'https://www.eba.europa.eu'
homepage_url = base_url + '/homepage'

response = polite_request(homepage_url)
soup = BeautifulSoup(response.text, 'html.parser')
articles = soup.find_all(class_="teaser-columns")


In [38]:
print(soup)

<!DOCTYPE html>

<html dir="ltr" lang="en">
<head>
<meta charset="utf-8"/>
<script>var _paq = _paq || [];(function(){var u=(("https:" == document.location.protocol) ? "https://analytics.eba.europa.eu/" : "https://analytics.eba.europa.eu/");_paq.push(["setSiteId", "1"]);_paq.push(["setTrackerUrl", u+"matomo.php"]);_paq.push(["setDoNotTrack", 1]);_paq.push(['requireCookieConsent']);
_paq.push(['requireConsent']);if (!window.matomo_search_results_active) {_paq.push(["trackPageView"]);}_paq.push(["setIgnoreClasses", ["no-tracking","colorbox"]]);_paq.push(["enableLinkTracking"]);if (!document.cookie.includes('mtm_consent_removed')) {
  _paq.push(['setConsentGiven']);
  _paq.push(['setCookieConsentGiven']);
}var d=document,g=d.createElement("script"),s=d.getElementsByTagName("script")[0];g.type="text/javascript";g.defer=true;g.async=true;g.src=u+"matomo.js";s.parentNode.insertBefore(g,s);})();</script>
<script type="application/json">{"utility":"cck","url":"\/cookie-policy-for-eba-official-w

In [39]:
articles = soup.find_all(class_="teaser-columns")

In [40]:
articles

[<article class="teaser-columns" data-element="teaser-columns">
 <div class="teaser-columns__metadata" data-field="metadata">
 <div class="link-icon link-icon--flex link-icon--calendar">
     30 APRIL 2025
   </div>
 </div>
 <div class="teaser-columns__content">
 <h3 class="h5 teaser-columns__title" data-field="title"><a href="/publications-and-media/press-releases/eba-consults-draft-amending-technical-standards-factors-assessing-appropriateness-real-estate-risk" rel="bookmark">
 <span>The EBA consults on draft amending technical standards on factors assessing the appropriateness of real estate risk weights</span>
 </a></h3>
 <div class="teaser-columns__text" data-field="text">
 <p><p>The European Banking Authority (EBA) today launched a public consultation on its draft amending Regulatory Technical Standards (RTS) on the types of factors to be considered by national authorities in assessing the appropriateness of real estate risk weights. This review is driven by the revised Capital R

In [41]:
results = []
for article in articles[:3]:

    a_tag = article.find("a")
    title = a_tag.get_text(strip=True) if a_tag else "No title"
    relative_url = a_tag.get("href") if a_tag else ""
    url = base_url + relative_url if relative_url else "No URL"


    time_tag = article.find("time")
    pub_date = time_tag.get_text(strip=True) if time_tag else "No date"


    article_response = polite_request(url)
    article_soup = BeautifulSoup(article_response.text, 'html.parser')

    content_div = article_soup.find(class_="text-content") or article_soup.find("article") or article_soup.find("div", class_="content")
    main_content = content_div.get_text(separator="\n", strip=True) if content_div else "No content found"

    regulator = "EBA"

    results.append({
        "Title": title,
        "URL": url,
        "Publication date": pub_date,
        "Main content": main_content,
        "Regulator identifier": regulator
    })


for item in results:
    print("\n--- Article ---")
    for key, value in item.items():
        print(f"{key}: {value}")


--- Article ---
Title: The EBA consults on draft amending technical standards on factors assessing the appropriateness of real estate risk weights
URL: https://www.eba.europa.eu/publications-and-media/press-releases/eba-consults-draft-amending-technical-standards-factors-assessing-appropriateness-real-estate-risk
Publication date: No date
Main content: The EBA consults on draft amending technical standards on factors assessing the appropriateness of real estate risk weights
News & Press
30 April 2025
The European Banking Authority (EBA) today launched a public consultation on its draft amending Regulatory Technical Standards (RTS) on the types of factors to be considered by national authorities in assessing the appropriateness of real estate risk weights. This review is driven by the revised Capital Requirements Regulation (CRR 3), which confers a new mandate onto the EBA. The consultation runs until 30 May 2025.
Based on the assessment of the CRR3 changes to the treatment of exposure

In [49]:
data = []
for article in articles[:3]:
    a_tag = article.find("a")
    title = a_tag.get_text(strip=True) if a_tag else "No title"
    relative_url = a_tag.get("href") if a_tag else ""
    url = base_url + relative_url if relative_url else "No URL"
    
    metadata_div = article.find(class_="teaser-columns__metadata")
    pub_date = metadata_div.get_text(strip=True) if metadata_div else "No date"
    
    article_response = polite_request(url)
    article_soup = BeautifulSoup(article_response.text, 'html.parser')
    content_div = article_soup.find(class_="text-content") or article_soup.find("article") or article_soup.find("div", class_="content")
    main_content = content_div.get_text(separator="\n", strip=True) if content_div else "No content found"
    regulator = "EBA"
    data.append({
        "Title": title,
        "URL": url,
        "Publication date": pub_date,
        "Main content": main_content,
        "Regulator identifier": regulator
    })

In [50]:
df = pd.DataFrame(data, columns=["Title", "URL", "Publication date", "Main content", "Regulator identifier"])


In [52]:
df.to_csv("eba_articles.csv", index=False)

In [51]:
df

,Title,URL,Publication date,Main content,Regulator identifier
0,The EBA consults on draft amending technical s...,https://www.eba.europa.eu/publications-and-med...,30 APRIL 2025,The EBA consults on draft amending technical s...,EBA
1,The EBA issues criteria to determine when Cryp...,https://www.eba.europa.eu/publications-and-med...,25 APRIL 2025,The EBA issues criteria to determine when Cryp...,EBA
2,The EBA publishes key indicators on climate ri...,https://www.eba.europa.eu/publications-and-med...,25 APRIL 2025,The EBA publishes key indicators on climate ri...,EBA


# Federal Reserve System

In [1]:
from selenium import webdriver
from selenium.webdriver.safari.options import Options
from bs4 import BeautifulSoup
import time


options = Options()


driver = webdriver.Safari(options=options)


driver.get("https://www.federalreserve.gov/newsevents/pressreleases.htm")
time.sleep(3) 


html = driver.page_source
driver.quit()  




In [2]:

soup = BeautifulSoup(html, "html.parser")


articles = soup.select("div.row.ng-scope")

In [27]:
import random
titles = []
main_contents = []
dates = []
links = []
regulator_identifiers = []


for article in articles:

    title_tag = article.select_one("a.ng-binding")
    title = title_tag.get_text(strip=True) if title_tag else "N/A"
    link = "https://www.federalreserve.gov" + title_tag['href'] if title_tag else "N/A"


    content_tag = article.select_one("a.ng-binding")
    main_content = content_tag.get_text(strip=True) if content_tag else "N/A"

    date_tag = article.select_one("time.itemDate")
    date = date_tag.get_text(strip=True) if date_tag else "N/A"

    regulator_tag = article.select_one("em.ng-binding")
    regulator_identifier = regulator_tag.get_text(strip=True) if regulator_tag else "N/A"


    titles.append(title)
    main_contents.append(main_content)
    dates.append(date)
    links.append(link)
    regulator_identifiers.append(regulator_identifier)

random_indices = random.sample(range(len(titles)), 3) 


selected_articles = {
    'Article': [f"Article {i + 1}" for i in range(3)],
    'Title': [titles[idx] for idx in random_indices],
    'Main Content': [main_contents[idx] for idx in random_indices],
    'Date': [dates[idx] for idx in random_indices],
    'Link': [links[idx] for idx in random_indices],
    'Regulator Identifier': [regulator_identifiers[idx] for idx in random_indices]
}





In [28]:
selected_articles

{'Article': ['Article 1', 'Article 2', 'Article 3'],
 'Title': ['Federal Reserve Board announces approval of application by Capital One Financial Corporation to merge with Discover Financial Services and issues a consent order with Discover',
  'Federal Reserve Board announces termination of enforcement action with Lake Shore MHC and Lake Shore Bancorp Inc.',
  'Federal Reserve Board announces termination of enforcement action with former employee of The Marathon Bank'],
 'Main Content': ['Federal Reserve Board announces approval of application by Capital One Financial Corporation to merge with Discover Financial Services and issues a consent order with Discover',
  'Federal Reserve Board announces termination of enforcement action with Lake Shore MHC and Lake Shore Bancorp Inc.',
  'Federal Reserve Board announces termination of enforcement action with former employee of The Marathon Bank'],
 'Date': ['4/18/2025', '3/13/2025', '4/8/2025'],
 'Link': ['https://www.federalreserve.gov/new

In [30]:
federeal_reserve_service =  pd.DataFrame(selected_articles)

In [33]:
federeal_reserve_service.to_csv('federal_reserve_articles_with_regulator.csv', index = False )

# US Securities and  Exchange Comission

In [78]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.safari.service import Service
from selenium.webdriver.safari.options import Options
from bs4 import BeautifulSoup
import time


options = Options()
options.headless = True  
driver = webdriver.Safari()

base_url = "https://www.sec.gov"
main_url = f"{base_url}/newsroom/press-releases"
driver.get(main_url)
time.sleep(3)

html = driver.page_source
driver.quit()

In [84]:
soup = BeautifulSoup(html, "html.parser")



In [105]:
rows = soup.select("tr.pr-list-page-row")[:3]

In [89]:
rows

[<tr class="pr-list-page-row">
 <td class="views-field views-field-field-publish-date is-active" headers="view-field-publish-date-table-column"> <time class="datetime" datetime="2025-04-29T12:02:23Z">April 29, 2025</time>
 </td>
 <td class="views-field views-field-field-display-title" headers="view-field-display-title-table-column"> <a href="/newsroom/press-releases/2025-71" hreflang="en">SEC Charges Three Texans with Defrauding Investors in $91 Million Ponzi Scheme</a>
 </td>
 <td class="views-field views-field-field-release-number" headers="view-field-release-number-table-column">2025-71            </td>
 </tr>,
 <tr class="pr-list-page-row">
 <td class="views-field views-field-field-publish-date is-active" headers="view-field-publish-date-table-column"> <time class="datetime" datetime="2025-04-28T12:55:17Z">April 28, 2025</time>
 </td>
 <td class="views-field views-field-field-display-title" headers="view-field-display-title-table-column"> <a href="/newsroom/press-releases/2025-70" 

In [106]:
from selenium import webdriver
from selenium.webdriver.safari.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time

options = Options()
driver = webdriver.Safari(options=options)

base_url = "https://www.sec.gov"
headers = {'User-Agent': 'Mozilla/5.0'}


driver.get(f"{base_url}/newsroom/press-releases")


WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.TAG_NAME, "html"))
)

page_source = driver.page_source
soup = BeautifulSoup(page_source, "html.parser")


rows = soup.select("tr.pr-list-page-row")[:3]

data = []
for row in rows:
    title_tag = row.select_one("td.views-field-field-display-title a")
    date_tag = row.select_one("time.datetime")

    if title_tag and date_tag:
        title = title_tag.get_text(strip=True)
        relative_link = title_tag["href"]
        link = base_url + relative_link
        date = date_tag.get_text(strip=True)

        try:
            driver.get(link)
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "html"))
            )
            article_soup = BeautifulSoup(driver.page_source, "html.parser")
            
            content_div = article_soup.select_one("div.field.field--name-body")
            main_content = content_div.get_text(separator=" ", strip=True) if content_div else "N/A"
            
            press_release_number = article_soup.select_one("div.field--name-field-release-number .field__item")
            release_number = press_release_number.get_text(strip=True) if press_release_number else "N/A"
            
            release_date = article_soup.select_one("div.field--name-dynamic-twig-fieldnode-press-release-lead-in p")
            release_date_text = release_date.get_text(strip=True) if release_date else "N/A"
            
            for_immediate_release = article_soup.select_one(".news__press_release__immediate_release p")
            immediate_release = for_immediate_release.get_text(strip=True) if for_immediate_release else "N/A"
            
        except Exception as e:
            main_content = f"Error fetching content: {e}"
            release_number = release_date_text = immediate_release = "N/A"

        data.append({
            "Title": title,
            "Date": date,
            "Link": link,
            "Main Content": main_content,
            "Press Release Number": release_number,
            "Release Date": release_date_text,
            "For Immediate Release": immediate_release,
            "Regulator Identifier": "SEC"
        })


driver.quit()


for item in data:
    print(f"{item['Date']} - {item['Title']}\n{item['Link']}\n{item['Main Content'][:300]}...\n")


April 29, 2025 - SEC Charges Three Texans with Defrauding Investors in $91 Million Ponzi Scheme
https://www.sec.gov/newsroom/press-releases/2025-71
The Securities and Exchange Commission today announced charges against Dallas-Fort Worth residents Kenneth W. Alexander II, Robert D. Welsh, and Caedrynn E. Conner for operating a Ponzi scheme that raised at least $91 million from more than 200 investors. According to the SEC’s complaint, between ap...

April 28, 2025 - SEC Publishes New Market Data, Analysis, and Visualizations
https://www.sec.gov/newsroom/press-releases/2025-70
The Securities and Exchange Commission’s Division of Economic and Risk Analysis (DERA) has published new data and analysis on the key market areas of public issuers, exempt offerings, Commercial Mortgage-Backed Securities (CMBS), Asset-Backed Securities (ABS), money market funds, and security-based ...

April 22, 2025 - SEC Charges PGI Global Founder with $198 Million Crypto Asset and Foreign Exchange Fraud Scheme


In [107]:
data_SEC = pd.DataFrame(data)

In [108]:
data_SEC

,Title,Date,Link,Main Content,Press Release Number,Release Date,For Immediate Release,Regulator Identifier
0,SEC Charges Three Texans with Defrauding Inves...,"April 29, 2025",https://www.sec.gov/newsroom/press-releases/20...,The Securities and Exchange Commission today a...,2025-71,"Washington D.C., April 29, 2025 —",For Immediate Release,SEC
1,"SEC Publishes New Market Data, Analysis, and V...","April 28, 2025",https://www.sec.gov/newsroom/press-releases/20...,The Securities and Exchange Commission’s Divis...,2025-70,"Washington D.C., April 28, 2025 —",For Immediate Release,SEC
2,SEC Charges PGI Global Founder with $198 Milli...,"April 22, 2025",https://www.sec.gov/newsroom/press-releases/20...,The Securities and Exchange Commission today c...,2025-69,"Washington D.C., April 22, 2025 —",For Immediate Release,SEC


In [109]:
data_SEC.to_csv('US_Securities_and_Exchange_Comission.csv', index = False )